In [1]:
import numpy as np
import cvxpy as cp

In [2]:
def DNM(f_grad_hess, x0, A, b, tol=0.01, max_iter=1000, alpha=0.25, beta=0.5):
    """
    Демпфированный метод Ньютона для задач с линейными равенствами (Ax = b).
    params:
      f_grad_hess: функция, возвращающая (f, grad, hess) в точке x
      x0: начальная точка (должна удовлетворять Ax = b)
      A, b: параметры ограничения Ax = b
      alpha: параметр для backtracking line search (ожидаемое убывание)
      beta: параметр для backtracking line search (множитель уменьшения шага)
    """
    x = x0.copy().astype(float)
    n = len(x)
    m = A.shape[0]

    for i in range(max_iter):
        f, grad, hess = f_grad_hess(x)

        kkt_matrix = np.block([
            [hess, A.T],
            [A, np.zeros((m, m))]
        ])
        rhs = np.concatenate([-grad, np.zeros(m)])

        try:
            sol = np.linalg.solve(kkt_matrix, rhs)
        except np.linalg.LinAlgError:
            print("Singular Hessian!")
            break

        step = sol[:n]

        f_decrement_sq = -grad.dot(step)
        if f_decrement_sq < 0:
             break
        if f_decrement_sq / 2.0 <= tol:
            return x, f, i, sol[n:]

        t = 1.0
        while True:
            x_next = x + t * step
            try:
                f_next, _, _ = f_grad_hess(x_next)
                if not np.isnan(f_next) and f_next <= f + alpha * t * grad.dot(step):
                    break
            except:
                pass
            t *= beta
            if t < 1e-15: break

        x = x_next

    return x, f, max_iter, sol[n:]

In [3]:
def test_DNM():
    np.random.seed(42)
    n, m = 3, 5
    pi = np.ones(m) / m
    P = np.random.uniform(0.7, 1.5, (m, n))

    def primal_f(x):
        returns = P @ x
        if np.any(returns <= 1e-9): return np.nan, None, None
        f = -np.sum(pi * np.log(returns))
        grad = -P.T @ (pi / returns)
        hess = (P.T * (pi / (returns**2))) @ P
        return f, grad, hess

    A_p, b_p = np.ones((1, n)), np.array([1.0])
    x0_p = np.ones(n) / n
    x_opt_dnm, f_p_dnm, iters_p, nu_opt_dnm = DNM(primal_f, x0_p, A_p, b_p, tol=1e-8, alpha=0.3, beta=0.1)

    print(f"Количество итераций решения прямой задачи: {iters_p}")

    def dual_f(lam):
        if np.any(lam <= 1e-9): return np.nan, None, None
        f = -np.sum(pi * np.log(lam / pi)) - 1
        grad = -pi / lam
        hess = np.diag(pi / (lam**2))
        return f, grad, hess

    A_d, b_d = P.T, np.ones(n)
    lam0 = np.linalg.pinv(A_d) @ b_d
    if np.any(lam0 <= 0): lam0 = np.ones(m) / m
    lam_opt_dnm, f_d_dnm, iters_d, _ = DNM(dual_f, lam0, A_d, b_d, tol=1e-8, alpha=0.3, beta=0.1)

    print(f"Количество итераций решения двойственной задачи: {iters_d}")

    x_cp = cp.Variable(n)
    prob_p = cp.Problem(cp.Minimize(-pi @ cp.log(P @ x_cp)), [cp.sum(x_cp) == 1])
    prob_p.solve()

    nu_cp = prob_p.constraints[0].dual_value

    print("Прямая задача:")
    print(f"DNM: {x_opt_dnm}")
    print(f"CVX: {x_cp.value}")
    print(f"Разница: {np.linalg.norm(x_opt_dnm - x_cp.value):.2e}")

    print("Множитель Лагранжа:")
    print(f"DNM: {nu_opt_dnm[0]:.4f}")
    print(f"CVX: {nu_cp:.4f}")

    print("Двойственная задача:")
    lam_theory = pi / (P @ x_opt_dnm)
    print(f"DNM: {lam_opt_dnm}")
    print(f"Теоретическое значение (через прямую задачу): {lam_theory}")
    print(f"Разница: {np.linalg.norm(lam_opt_dnm - lam_theory):.2e}")

    assert np.allclose(x_opt_dnm, x_cp.value, atol=1e-3), "DNM не совпал с CVX"
    assert np.allclose(nu_opt_dnm, 1.0, atol=1e-5), "Множитель Лагранжа не равен 1"
    assert np.abs(A_p @ x_opt_dnm - b_p) < 1e-9, "DNM не удовлетворяет условию равенства"
    print("Тест пройден успешно")

In [4]:
test_DNM()

Количество итераций решения прямой задачи: 3
Количество итераций решения двойственной задачи: 2
Прямая задача:
DNM: [ 0.39851452 -0.53217617  1.13366166]
CVX: [ 0.39855486 -0.53205505  1.13350019]
Разница: 2.06e-04
Множитель Лагранжа:
DNM: 1.0000
CVX: 1.0000
Двойственная задача:
DNM: [0.18544356 0.20705889 0.2234865  0.11131249 0.19233408]
Теоретическое значение (через прямую задачу): [0.18543991 0.20705825 0.22348368 0.11132028 0.19232742]
Разница: 1.13e-05
Тест пройден успешно
